In [27]:
from web3 import Web3
import json
KEY = "8d900ed9a1974cfba7acafa2b98544b0"
# Set up your Web3 provider
INFURA_URL = f"https://mainnet.infura.io/v3/{KEY}"
web3 = Web3(Web3.HTTPProvider(INFURA_URL))

# Contract address and ABI
POOL_ADDRESS = "0x02950460e2b9529d0e00284a5fa2d7bdf3fa4d72"
POOL_ABI = [
    {
        "anonymous": False,
        "inputs": [
            {"indexed": False, "name": "fee", "type": "uint256"},
            {"indexed": False, "name": "admin_fee", "type": "uint256"}
        ],
        "name": "NewFee",
        "type": "event"
    },
    {
        "anonymous": False,
        "inputs": [
            {"indexed": False, "name": "old_A", "type": "uint256"},
            {"indexed": False, "name": "new_A", "type": "uint256"},
            {"indexed": False, "name": "initial_time", "type": "uint256"},
            {"indexed": False, "name": "future_time", "type": "uint256"}
        ],
        "name": "RampA",
        "type": "event"
    }
]

#read json from file
with open('notebooks/text.json', 'r') as f:
    POOL_ABI = json.load(f)



In [28]:
POOL_ABI = json.loads(POOL_ABI["result"])

In [29]:
# POOL_ABI

In [30]:

contract = web3.eth.contract(address=web3.to_checksum_address(POOL_ADDRESS), abi=POOL_ABI)

# Define a function to fetch and parse events
def fetch_events(event_name, from_block, to_block):
    try:
        # Get the event object
        event = contract.events[event_name]
        # Fetch logs
        logs = event.get_logs(fromBlock=from_block, toBlock=to_block)
        # Parse logs

        return logs
    except Exception as e:
        print(f"Error fetching {event_name} events: {e}")
        return []

# Specify block range for querying
START_BLOCK = 0  # Replace with the block number where the pool was deployed
END_BLOCK = web3.eth.block_number

# Fetch NewFee and RampA events
new_fee_events = fetch_events("NewFee", START_BLOCK, END_BLOCK)
ramp_a_events = fetch_events("RampA", START_BLOCK, END_BLOCK)



In [31]:
contract.functions.balances(0).call()

latest


8457789582115141970169759

In [32]:
ramp_a_events

(AttributeDict({'args': AttributeDict({'old_A': 20000,
   'new_A': 80000,
   'initial_time': 1718224559,
   'future_time': 1718829078}),
  'event': 'RampA',
  'logIndex': 422,
  'transactionIndex': 220,
  'transactionHash': HexBytes('0xb1a056c767cca093cc8d28f224e1648ff1e9da727000e265f8676296448af143'),
  'address': '0x02950460E2b9529D0E00284A5fA2d7bDF3fA4d72',
  'blockHash': HexBytes('0xd674d9c177bb96b85caf6bd24e77697dbc1f07770d27f8314979dbd07c9209a1'),
  'blockNumber': 20078172}),)

In [40]:
JUN12 = 1718224559
MAG12 = 1715513547
JUN19 = 1718829078
JUL19 = 1721388747
POOL = "0x02950460e2b9529d0e00284a5fa2d7bdf3fa4d72"


In [39]:
import requests
import pandas as pd
from datetime import datetime

def fetch(pool, since, until):
    # URL per i dati
    url = f"https://prices.curve.fi/v1/volume/usd/ethereum/{pool}?interval=day&start={since}&end={until}"

    # Scarica i dati dalla URL
    response = requests.get(url)
    response.raise_for_status()  # Controlla errori HTTP

    data = response.json()

    # Estrai i dati rilevanti
    data_list = data.get("data", [])

    # Converti in DataFrame
    df = pd.DataFrame(data_list)

    # Aggiungi una colonna per data leggibile (opzionale)
    df['date'] = pd.to_datetime(df['timestamp'], unit='s')

    # Calcola la media storica (30 giorni) di volume e fee
    return df




In [45]:
df = fetch(POOL, JUN19, JUL19)
display(df)
display(df["fees"].mean())
display(df["volume"].mean())

,timestamp,volume,fees,date
0,1718755200,1.425121e+06,142.513798,2024-06-19
1,1718841600,5.372646e+06,547.832146,2024-06-20
2,1718928000,6.050819e+06,605.149680,2024-06-21
3,1719014400,1.181711e+07,1179.295854,2024-06-22
4,1719100800,1.221836e+07,1221.783996,2024-06-23
5,1719187200,2.703950e+07,2704.003005,2024-06-24
6,1719273600,1.723954e+07,1724.008565,2024-06-25
7,1719360000,1.914694e+07,1914.838476,2024-06-26
8,1719446400,1.428369e+07,1428.362657,2024-06-27
9,1719532800,1.348103e+07,1348.084785,2024-06-28


1523.8166159341417

15228999.661912253